# Explainable AI Analysis

This notebook performs explainability analysis on a the trained transformer model using Integrated Gradients. It identifies the most important words and features that contribute to model predictions.

In [ ]:
import torch
import pandas as pd
import numpy as np
import datetime
import os
import logging
import random
import matplotlib.pyplot as plt
import re

from collections import defaultdict, Counter
from captum.attr import IntegratedGradients
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, set_seed


### Configuration and Seed Setting

In [ ]:
SEED = 42

def set_all_seeds(seed: int = SEED, deterministic: bool = False):
    """
    Set all random seeds for reproducibility across Python, NumPy, PyTorch, and Transformers.

    Args:
        seed: Random seed value to use for all random number generators.
        deterministic: If True, enables deterministic CUDA operations. This may reduce performance
                      but ensures full reproducibility on GPU. Note that some operations may not
                      have deterministic implementations.

    Returns:
        None
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    set_seed(seed)

    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.use_deterministic_algorithms(True)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(SEED, deterministic=False)

### Error Logging Setup

In [ ]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

log_file = os.path.join(log_dir, "error_log.csv")

logging.basicConfig(level=logging.ERROR,
                    format="%(asctime)s,%(levelname)s,%(message)s",
                    datefmt="%Y-%m-%d %H:%M:%S")

def log_error(error_message, model_name="Unknown"):
    """
    Log error messages to a CSV file with timestamp and model information.

    Args:
        error_message: The error message or exception details to log.
        model_name: Name or identifier of the model where the error occurred.

    Returns:
        None
    """

    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if not os.path.exists(log_file):
        df = pd.DataFrame(columns=["timestamp", "model_name", "error_message"])
        df.to_csv(log_file, index=False)

    df = pd.DataFrame([[timestamp, model_name, error_message]],
                      columns=["timestamp", "model_name", "error_message"])
    df.to_csv(log_file, mode="a", header=False, index=False)

### Paths and Configuration

In [ ]:
MODEL_PATH = "../models/FacebookAI_roberta-base"
OUTPUT_DIR = "../models/XAI_RobertaBase"

DATA_PATH = "../data/input/ProcessDataset.csv"
DATA_ENCODING = "latin1"
DATA_SEP = ";"

LABEL_COL = "GDPR-criticality"
TEXT_COL = "Concatenation"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Loading and Preparation

In [ ]:
raw_data = pd.read_csv(DATA_PATH, sep=DATA_SEP, encoding=DATA_ENCODING)
raw_data = raw_data.dropna(axis=1)
input_data = raw_data.copy(deep=True)

input_data.rename(columns={TEXT_COL:'text', LABEL_COL:'labels'}, inplace=True)
X_train, X_vtest, y_train, y_vtest = train_test_split(input_data['text'], input_data['labels'],
                                                      test_size=0.4,
                                                      random_state=SEED)
X_test, X_val, y_test, y_val = train_test_split(X_vtest, y_vtest,
                                                test_size=0.5,
                                                random_state=SEED)

datasets = DatasetDict({
    "train": Dataset.from_pandas(pd.concat([X_train, y_train], axis=1)),
    "test": Dataset.from_pandas(pd.concat([X_test, y_test], axis=1)),
    "val": Dataset.from_pandas(pd.concat([X_val, y_val], axis=1))
    })

### Model and Tokenizer Setup

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

def preprocess_function(examples):
    """
    Tokenize text examples for transformer model input.

    Args:
        examples: Dictionary containing 'text' field with input texts.

    Returns:
        Dictionary with tokenized inputs (input_ids, attention_mask, etc.).
    """
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )
tokenized_data = datasets.map(preprocess_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=2,
    torch_dtype="auto",
    device_map="cuda" if torch.cuda.is_available() else None,
    ignore_mismatched_sizes=True
).to(device)


def compute_metrics(eval_pred):
    """
    Compute evaluation metrics from model predictions.

    Args:
        eval_pred: Tuple of (logits, labels) from model evaluation.

    Returns:
        Dictionary containing accuracy and recall scores.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "recall": recall_score(labels, preds, average="binary", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    num_train_epochs=3,
    load_best_model_at_end=True,
    metric_for_best_model="recall",
    seed=SEED,
    data_seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["val"],
    compute_metrics=compute_metrics
)

### Running Prediction

In [ ]:
test_predictions = trainer.predict(tokenized_data["test"])
logits = test_predictions.predictions

probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
p_crit = probs[:, 1]

y_pred = np.argmax(logits, axis=1)
y_true = np.array(tokenized_data["test"]["labels"])

### Integrated Gradients Implementation

In [ ]:
def _predict_proba_single(text: str):
    """
    Compute class probabilities for a single input using the model.

    Args:
        input_ids: Token IDs tensor of shape (seq_len,).
        attention_mask: Attention mask tensor of shape (seq_len,).

    Returns:
        Array of class probabilities of shape (num_classes,).
    """
    model.eval()
    with torch.no_grad():
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=False
        ).to(device)
        out = model(**enc)
        probs = torch.softmax(out.logits, dim=-1).squeeze(0).detach().cpu().numpy()
    return probs


def _forward_logits_for_target(inputs_embeds, attention_mask, target_class: int):
    """
    Forward pass through model returning logit for target class.

    Args:
        emb: Embedding tensor of shape (1, seq_len, hidden_dim).
        attention_mask: Attention mask tensor of shape (1, seq_len).
        target_class: Index of the target class.

    Returns:
        Logit value for the target class.
    """
    out = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
    return out.logits[:, target_class]


def _aggregate_roberta_tokens_to_words(tokens, scores):
    """
    Aggregate RoBERTa subword tokens into complete words with summed scores.

    RoBERTa uses byte-level BPE tokenization where subwords are prefixed with 'Ġ' (space).
    This function merges subword tokens into complete words and sums their attribution scores.

    Args:
        tokens: List of RoBERTa tokens (including special tokens and subwords).
        token_scores: Array of attribution scores for each token.

    Returns:
        List of (word, score) tuples with aggregated scores.
    """
    words = []
    current_word = ""
    current_score = 0.0

    for tok, sc in zip(tokens, scores):
        # skip special tokens
        if tok in tokenizer.all_special_tokens:
            continue

        # RoBERTa word boundary: token starts with Ġ
        if tok.startswith("Ġ"):
            # flush previous
            if current_word != "":
                words.append((current_word, current_score))
            current_word = tok[1:]  # remove Ġ
            current_score = float(sc)
        else:
            # continuation of same word
            current_word += tok
            current_score += float(sc)

    if current_word != "":
        words.append((current_word, current_score))

    # Clean up weird empties
    words = [(w, s) for w, s in words if w and not w.isspace()]
    return words


def integrated_gradients_explain_text(
        text: str,
        target_class: int | None = None,
        n_steps: int = 80,
        internal_batch_size: int = 16,
    ):
    """
    Compute Integrated Gradients attribution scores for input text.

    Integrated Gradients is an attribution method that assigns importance scores to input features
    by integrating gradients along a path from a baseline (zero embedding) to the actual input.

    Args:
        text: Input text to explain.
        target_class: Class index to compute attributions for (0 or 1).
        n_steps: Number of interpolation steps for the integral approximation.

    Returns:
        List of (word, attribution_score) tuples sorted by absolute attribution value.
    """
    model.eval()

    # tokenize to fixed length? for IG better to avoid max_length padding unless needed
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=False,
        add_special_tokens=True
    )
    input_ids = enc["input_ids"].to(device)  # (1, seq_len)
    attention_mask = enc["attention_mask"].to(device)  # (1, seq_len)

    # choose target
    probs = _predict_proba_single(text)
    pred_class = int(np.argmax(probs))
    if target_class is None:
        target_class = pred_class

    # embeddings
    emb_layer = model.get_input_embeddings()
    inputs_embeds = emb_layer(input_ids)  # (1, seq_len, hidden)

    # baseline = PAD embeddings
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        # fallback for tokenizers without pad_token_id
        pad_id = tokenizer.eos_token_id or tokenizer.sep_token_id or tokenizer.unk_token_id
    baseline_ids = torch.full_like(input_ids, fill_value=pad_id)
    baseline_embeds = emb_layer(baseline_ids)

    # IG
    ig = IntegratedGradients(lambda emb, mask: _forward_logits_for_target(emb, mask, target_class))
    attributions, delta = ig.attribute(
        inputs=inputs_embeds,
        baselines=baseline_embeds,
        additional_forward_args=(attention_mask,),
        n_steps=n_steps,
        internal_batch_size=internal_batch_size,
        return_convergence_delta=True
    )

    # token scores: sum over hidden dim
    token_scores = attributions.sum(dim=-1).squeeze(0).detach().cpu().numpy()  # (seq_len,)

    # tokens
    ids = input_ids.squeeze(0).detach().cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(ids)

    # word aggregation (RoBERTa-specific)
    word_pairs = _aggregate_roberta_tokens_to_words(tokens, token_scores)

    return {
        "text": text,
        "probs": probs,
        "pred_class": pred_class,
        "target_class": target_class,
        "convergence_delta": float(delta.detach().cpu().item()) if torch.is_tensor(delta) else float(delta),
        "tokens": tokens,
        "token_scores": token_scores,
        "word_pairs": word_pairs,  # list[(word, score)]
    }


def topk_pairs(pairs, k=20):
    """
    Select top-k word-score pairs by absolute score value.

    Args:
        pairs: List of (word, score) tuples.
        k: Number of top pairs to return.

    Returns:
        List of top-k (word, score) tuples sorted by absolute score (descending).
    """
    return sorted(pairs, key=lambda x: abs(x[1]), reverse=True)[:k]

### Excecute IG and Set up general unpolished DataFrame for SAMPLE_SIZE

In [ ]:
# Build a DataFrame similar to your df_expl idea:
test_texts = datasets["test"]["text"]  # original texts
test_labels = np.array(datasets["test"]["labels"])

# choose indices: e.g. 50 random test samples for XAI
rng = np.random.default_rng(SEED)
SAMPLE_SIZE = 50
idxs = rng.choice(len(test_texts), size=min(SAMPLE_SIZE, len(test_texts)), replace=False)

rows = []
for i in idxs:
    t = test_texts[int(i)]
    true_y = int(test_labels[int(i)])

    # Option A: explain predicted class
    res_pred = integrated_gradients_explain_text(t, target_class=None)

    # Option B (often useful in GDPR criticality): explain class 1 always
    res_crit = integrated_gradients_explain_text(t, target_class=1)

    rows.append({
        "idx": int(i),
        "y_true": true_y,
        "pred_class": int(res_pred["pred_class"]),
        "p_class0": float(res_pred["probs"][0]),
        "p_class1": float(res_pred["probs"][1]),
        "delta_pred": float(res_pred["convergence_delta"]),
        "delta_crit": float(res_crit["convergence_delta"]),
        "top_words_pred": topk_pairs(res_pred["word_pairs"], k=25),
        "top_words_crit": topk_pairs(res_crit["word_pairs"], k=25),
    })

df_ig = pd.DataFrame(rows)
df_ig.head()


### Aggregated IG Information on Word Level

In [ ]:
def aggregate_word_scores(df, col="top_words_crit"):
    """
    Aggregates word scores across samples.
    col: "top_words_crit" (class 1) or "top_words_pred"
    Returns DataFrame with word, sum_abs, sum_signed, count
    """
    agg_abs = defaultdict(float)
    agg_signed = defaultdict(float)
    cnt = defaultdict(int)

    for pairs in df[col]:
        for w, s in pairs:
            w_norm = w.lower().strip()
            # optional cleanup: keep only words with letters/numbers
            w_norm = re.sub(r"^\W+|\W+$", "", w_norm)
            if not w_norm:
                continue
            agg_abs[w_norm] += abs(float(s))
            agg_signed[w_norm] += float(s)
            cnt[w_norm] += 1

    out = pd.DataFrame({
        "word": list(agg_abs.keys()),
        "sum_abs": [agg_abs[w] for w in agg_abs.keys()],
        "sum_signed": [agg_signed[w] for w in agg_signed.keys()],
        "count": [cnt[w] for w in cnt.keys()],
    }).sort_values("sum_abs", ascending=False)

    return out

df_global_crit = aggregate_word_scores(df_ig, col="top_words_crit")
df_global_pred = aggregate_word_scores(df_ig, col="top_words_pred")


### IG result Cleanu-Up on Word Level

In [ ]:
def normalize_word(w: str) -> str:
    w = w.lower().strip()
    w = re.sub(r"^\W+|\W+$", "", w)  # punctuation ab
    return w

def collapse_pairs(pairs):
    d = defaultdict(float)
    for w, s in pairs:
        wn = normalize_word(w)
        if wn:
            d[wn] += float(s)
    # sortiere nach absoluter Stärke
    return sorted(d.items(), key=lambda x: abs(x[1]), reverse=True)

df_ig["top_words_crit_collapsed"] = df_ig["top_words_crit"].apply(collapse_pairs)


### DataFrame Slicing in Conf-Mat Results

In [ ]:
df_ig_tp = df_ig[((df_ig["y_true"]==1)  & (df_ig["pred_class"]==1))]
df_ig_tn = df_ig[((df_ig["y_true"]==0)  & (df_ig["pred_class"]==0))]
df_ig_fp = df_ig[((df_ig["y_true"]==0)  & (df_ig["pred_class"]==1))]
df_ig_fn = df_ig[((df_ig["y_true"]==1)  & (df_ig["pred_class"]==0))]
df_ig_fp.iloc[0]


### Aggregate Lokal Results on Global Results

In [ ]:
def global_word_stats(
    df,
    col="top_words_crit_collapsed",
    top_k_per_doc=25
):
    """
    Compute global statistics for words across all explanations.

    Args:
        all_explanations: List of word-score lists from all text explanations.
        top_n: Number of top words to return.

    Returns:
        DataFrame with columns: word, sum_abs, sum_signed, doc_freq.
    """
    sum_abs = defaultdict(float)
    sum_signed = defaultdict(float)
    doc_freq = Counter()

    for pairs in df[col]:
        for w, s in pairs[:top_k_per_doc]:
            sum_abs[w] += abs(float(s))
            sum_signed[w] += float(s)
            doc_freq[w] += 1

    out = pd.DataFrame({
        "word": list(sum_abs.keys()),
        "sum_abs": [sum_abs[w] for w in sum_abs],
        "sum_signed": [sum_signed[w] for w in sum_signed],
        "doc_freq": [doc_freq[w] for w in sum_abs],
    }).sort_values("sum_abs", ascending=False)

    return out


### Define Conf-Mat-Elements on Global Level

In [ ]:
df_tp_global = global_word_stats(df_ig_tp)
df_fp_global = global_word_stats(df_ig_fp)
df_fn_global = global_word_stats(df_ig_fn)
df_tn_global = global_word_stats(df_ig_tn)

### Cross Comparisson Crit Words Conf-Mat

In [ ]:
TOP_N = 20

tp_words = set(df_tp_global.head(TOP_N)["word"])
fp_words = set(df_fp_global.head(TOP_N)["word"])
fn_words = set(df_fn_global.head(TOP_N)["word"])

df_compare_words = pd.DataFrame({
    "TP_top_words": pd.Series(list(tp_words)),
    "FP_top_words": pd.Series(list(fp_words)),
    "FN_top_words": pd.Series(list(fn_words)),
})

df_compare_words

### Cross Comparisson Crit Values Conf-Mat

In [ ]:
def score_diff(df_a, df_b, top_n=30):
    a = df_a.set_index("word")["sum_abs"]
    b = df_b.set_index("word")["sum_abs"]

    diff = (a - b).dropna().sort_values(ascending=False)
    return diff.head(top_n)

# TP vs FP
score_diff(df_tp_global, df_fp_global, top_n=20)

# TP vs FN
score_diff(df_tp_global, df_fn_global, top_n=20)


### Word Level Clean Up for Plotting

In [ ]:
XAI_STOP_WORDS = {
    "using", "in", "the", "context", "of", "no", "a", "to"
}
XAI_STOP_PHRASES = [
    "in the context of",
    "on the basis of",
    "with respect to",
    "in accordance with",
    "for the purpose of",
    "in order to",
    "as well as",
    "such as",
    "based on",
    "in relation to",
]

def filter_words_and_phrases(pairs, stop_words, stop_phrases):
    """
    pairs: list[(word, score)]
    """
    cleaned = []

    # Normalisierte Stop-Phrasen
    stop_phrases_norm = [p.lower().strip() for p in stop_phrases]

    for w, s in pairs:
        w_norm = normalize_word(w)

        # Einzelwort-Filter
        if w_norm in stop_words:
            continue

        cleaned.append((w_norm, s))

    # Phrase-Filter (nachträglich)
    words_only = [w for w, _ in cleaned]
    scores_only = [s for _, s in cleaned]

    to_remove = set()
    for phrase in stop_phrases_norm:
        p_tokens = phrase.split()
        p_len = len(p_tokens)

        for i in range(len(words_only) - p_len + 1):
            if words_only[i:i+p_len] == p_tokens:
                to_remove.update(range(i, i+p_len))

    final = [
        (words_only[i], scores_only[i])
        for i in range(len(words_only))
        if i not in to_remove
    ]

    return final


def apply_xai_cleanup(df, col_in="top_words_crit_collapsed", col_out="top_words_clean"):
    df = df.copy()

    df[col_out] = df[col_in].apply(
        lambda pairs: filter_words_and_phrases(
            pairs,
            stop_words=XAI_STOP_WORDS,
            stop_phrases=XAI_STOP_PHRASES
        )
    )
    return df


df_ig_tp_clean = apply_xai_cleanup(df_ig_tp)
df_ig_fp_clean = apply_xai_cleanup(df_ig_fp)
df_ig_fn_clean = apply_xai_cleanup(df_ig_fn)
df_ig_tn_clean = apply_xai_cleanup(df_ig_tn)

df_tp_global_clean = global_word_stats(df_ig_tp_clean, col="top_words_clean")
df_fp_global_clean = global_word_stats(df_ig_fp_clean, col="top_words_clean")
df_fn_global_clean = global_word_stats(df_ig_fn_clean, col="top_words_clean")
df_tn_global_clean = global_word_stats(df_ig_tn_clean, col="top_words_clean")


### Visualize XAI Results on Global Word Level

In [ ]:
METRICS = {
    "TP": df_tp_global_clean,
    "FP": df_fp_global_clean,
    "FN": df_fn_global_clean,
    "TN": df_tn_global_clean,
    "ALL": pd.concat([
        df_tp_global_clean,
        df_fp_global_clean,
        df_fn_global_clean,
        df_tn_global_clean
    ])
    .groupby("word", as_index=False)
    .agg({
        "sum_abs": "sum",
        "sum_signed": "sum",
        "doc_freq": "sum"
    })
    .sort_values("sum_abs", ascending=False)
}

topN = 20

for name, df_metric in METRICS.items():
    tmp = df_metric.head(topN).iloc[::-1]

    plt.figure(figsize=(8, 6))
    plt.barh(tmp["word"], tmp["sum_abs"])
    plt.xlim(0, 7)
    plt.xlabel("Aggregated |IG|")
    plt.ylabel("Word")
    plt.title(f"Most relevant words – {name} (Integrated Gradients)")
    plt.tight_layout()
    plt.savefig('../figures/{name}.svg'.format(name=name), dpi=600)
    plt.show()